In [5]:
#2.1
def calculate_conv_output():
    # 输入参数
    in_channels = 3
    in_h = in_w = 32
    num_kernels = 16
    kernel_size = 5
    padding = 2
    stride = 2
    
    # 输出尺寸计算
    out_h = (in_h + 2*padding - kernel_size) // stride + 1
    out_w = (in_w + 2*padding - kernel_size) // stride + 1
    out_channels = num_kernels
    
    print(f"2.1 卷积层输出尺寸: {out_channels} × {out_h} × {out_w}")
    
    # 单个输出像素的点乘次数
    mul_per_pixel = in_channels * kernel_size * kernel_size
    print(f"单个输出像素点乘次数: {mul_per_pixel} 次")
    
    return out_channels, out_h, out_w, mul_per_pixel

calculate_conv_output()

2.1 卷积层输出尺寸: 16 × 16 × 16
单个输出像素点乘次数: 75 次


(16, 16, 16, 75)

In [1]:
#2.2
import numpy as np

def max_pool2d(input, kernel_size, stride=None, padding=0):
    """
    手动实现二维最大池化前向传播
    input: numpy array, shape (C, H, W)
    kernel_size: int or tuple (kh, kw)
    stride: int or tuple (sh, sw), default = kernel_size
    padding: int or tuple (ph, pw)
    """
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    
    if stride is None:
        sh = sw = kh
    elif isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    C, H, W = input.shape
    
    # 添加 padding
    input_padded = np.pad(input, ((0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=0)
    
    H_out = (H + 2*ph - kh) // sh + 1
    W_out = (W + 2*pw - kw) // sw + 1
    
    output = np.zeros((C, H_out, W_out))
    
    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                h_start = i * sh
                h_end = h_start + kh
                w_start = j * sw
                w_end = w_start + kw
                region = input_padded[c, h_start:h_end, w_start:w_end]
                output[c, i, j] = np.max(region)
    
    return output

# 示例
if __name__ == "__main__":
    x = np.random.randn(3, 32, 32)
    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("Input shape:", x.shape)
    print("Output shape:", out.shape)

Input shape: (3, 32, 32)
Output shape: (3, 16, 16)


In [6]:
#3.1
def calculate_vgg_params():
    C = 1  # 用符号 C 表示通道数
    
    # 单个 5×5 卷积
    params_5x5 = C * C * 5 * 5
    print(f"3.1 单个5×5卷积参数量: {params_5x5} = 25C²")
    
    # 两个 3×3 卷积串联
    params_3x3_two = 2 * (C * C * 3 * 3)
    print(f"两个串联3×3卷积总参数量: {params_3x3_two} = 18C²")
    
    # 数值示例：C=16 时
    C_example = 16
    print(f"\n当 C={C_example} 时:")
    print(f"  5×5 参数量: {C_example * C_example * 5 * 5}")
    print(f"  两个3×3参数量: {2 * C_example * C_example * 3 * 3}")

calculate_vgg_params()

3.1 单个5×5卷积参数量: 25 = 25C²
两个串联3×3卷积总参数量: 18 = 18C²

当 C=16 时:
  5×5 参数量: 6400
  两个3×3参数量: 4608


In [1]:
#3.2
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)

# 示例
if __name__ == "__main__":
    block = NiNBlock(3, 16, kernel_size=3, stride=1, padding=1)
    x = torch.randn(1, 3, 32, 32)
    out = block(x)
    print("NiN Block output shape:", out.shape)

NiN Block output shape: torch.Size([1, 16, 32, 32])


In [7]:
#4.1
import numpy as np

def calculate_batch_norm():
    x = np.array([2, 4, 6, 8])
    gamma = 2
    beta = 1
    eps = 0
    
    # 计算均值和方差
    mu = np.mean(x)
    var = np.var(x)
    
    print(f"4.1 批量归一化计算")
    print(f"输入: {x}")
    print(f"均值 μ = {mu}")
    print(f"方差 σ² = {var}")
    
    # 标准化
    x_hat = (x - mu) / np.sqrt(var + eps)
    
    # 输出
    y = gamma * x_hat + beta
    
    print(f"标准化后 x̂ = {x_hat}")
    print(f"输出 y = {y}")
    
    return y

y = calculate_batch_norm()

4.1 批量归一化计算
输入: [2 4 6 8]
均值 μ = 5.0
方差 σ² = 5.0
标准化后 x̂ = [-1.34164079 -0.4472136   0.4472136   1.34164079]
输出 y = [-1.68328157  0.10557281  1.89442719  3.68328157]


In [2]:
#4.2
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity() if in_channels == out_channels else None
    
    def forward(self, x):
        identity = x
        if self.shortcut is not None:
            identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += identity
        out = self.relu(out)
        return out

# 示例
if __name__ == "__main__":
    block = Residual(3, 16, use_1x1conv=True, stride=2)
    x = torch.randn(1, 3, 32, 32)
    out = block(x)
    print("Residual block output shape:", out.shape)

Residual block output shape: torch.Size([1, 16, 16, 16])


In [8]:
#5.1
def fine_tuning_theory():
    print("5.1 微调理论问题")
    print("-" * 50)
    
    print("1. 为什么底层学习率小，顶层学习率大？")
    print("   - 底层特征（边缘、纹理等）具有通用性，在源数据集上已学得很好")
    print("   - 大幅调整底层会破坏已学到的通用特征")
    print("   - 顶层特征与具体类别相关，需要快速适应新任务")
    print("   - 因此：底层用小学习率（或冻结），顶层用大学习率")
    
    print("\n2. 目标数据集小且与源数据集相似时的策略：")
    print("   - 冻结大部分底层卷积层，只微调最后几层")
    print("   - 使用较小的全局学习率")
    print("   - 使用更强的正则化（dropout、权重衰减）")
    print("   - 使用图像增广（随机裁剪、翻转等）")
    print("   - 必要时可以只重新训练最后的全连接层")

fine_tuning_theory()

5.1 微调理论问题
--------------------------------------------------
1. 为什么底层学习率小，顶层学习率大？
   - 底层特征（边缘、纹理等）具有通用性，在源数据集上已学得很好
   - 大幅调整底层会破坏已学到的通用特征
   - 顶层特征与具体类别相关，需要快速适应新任务
   - 因此：底层用小学习率（或冻结），顶层用大学习率

2. 目标数据集小且与源数据集相似时的策略：
   - 冻结大部分底层卷积层，只微调最后几层
   - 使用较小的全局学习率
   - 使用更强的正则化（dropout、权重衰减）
   - 使用图像增广（随机裁剪、翻转等）
   - 必要时可以只重新训练最后的全连接层


In [3]:
#5.2
import torchvision.transforms as transforms

transform_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

# 示例使用
# from PIL import Image
# img = Image.open("test.jpg")
# img_tensor = transform_pipeline(img)

In [9]:
#6.1
def calculate_iou(box1, box2):
    """
    box: [x1, y1, x2, y2] 左上角和右下角坐标
    """
    # 交集
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    inter_area = max(0, x2_inter - x1_inter) * max(0, y2_inter - y1_inter)
    
    # 各自面积
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # 并集
    union_area = area1 + area2 - inter_area
    
    # IoU
    iou = inter_area / union_area if union_area > 0 else 0
    
    print(f"6.1 IoU 计算")
    print(f"真实框 A: {box1}")
    print(f"预测框 B: {box2}")
    print(f"交集面积: {inter_area}")
    print(f"并集面积: {union_area}")
    print(f"IoU = {inter_area}/{union_area} = {iou}")
    print(f"IoU 分数值: {iou:.6f}")
    
    return iou

box_A = [10, 10, 50, 50]
box_B = [30, 30, 70, 70]
iou = calculate_iou(box_A, box_B)

6.1 IoU 计算
真实框 A: [10, 10, 50, 50]
预测框 B: [30, 30, 70, 70]
交集面积: 400
并集面积: 2800
IoU = 400/2800 = 0.14285714285714285
IoU 分数值: 0.142857


In [4]:
#6.2
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, epsilon=0.1, reduction='mean'):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.epsilon = epsilon
        self.reduction = reduction
    
    def forward(self, pred, target):
        K = pred.size(-1)
        log_probs = F.log_softmax(pred, dim=-1)
        
        # 构建平滑标签
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.epsilon / (K - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1 - self.epsilon)
        
        loss = torch.sum(-true_dist * log_probs, dim=-1)
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# 示例
if __name__ == "__main__":
    criterion = LabelSmoothingCrossEntropy(epsilon=0.1)
    pred = torch.randn(4, 10)
    target = torch.tensor([1, 3, 5, 7])
    loss = criterion(pred, target)
    print("Label smoothing loss:", loss.item())

Label smoothing loss: 2.797252655029297
